In [26]:
import psutil
import platform

print("### Hardware Requirements ###")
print(f"CPU: {platform.processor()}")
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")

### Hardware Requirements ###
CPU: Intel64 Family 6 Model 170 Stepping 4, GenuineIntel
Total RAM: 34.01 GB


In [27]:
import tensorflow as tf
import pandas as pd
import numpy as np
import platform

print("### Software Requirements ###")
print(f"Python Version: {platform.python_version()}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Numpy Version: {np.__version__}")

### Software Requirements ###
Python Version: 3.9.13
TensorFlow Version: 2.12.0
Pandas Version: 1.5.0
Numpy Version: 1.23.5


In [28]:
import time
start_time = time.time()

# BioMapAI Tutorial

This tutorial will guide you through:
1. **Environment Setup**  
2. **Data Preparation**  
3. **Model Initialization and Training**  
4. **Prediction and Evaluation**  
5. **Score Conversion and Weight Adjustment**  

By the end, you should be able to train your own model on custom data.


## 1. Environment Setup

1. **Download or Clone** the repository/project that contains:
   - **BioMapAI.py** (the script with model classes and methods)
   - **example_data/** (containing your `train_data.csv` and `test_data.csv`)
   - A script or notebook for your tutorial (this file)

2. **Install Required Libraries** (if you haven’t already):
```bash
pip install numpy pandas tensorflow
```

3. **Import modules** in Python:
```python
import pandas as pd
import numpy as np
import random
import tensorflow as tf
import importlib.util
import os

# Clear any previous TensorFlow session
tf.keras.backend.clear_session()
```

4. **Function to import `BioMapAI.py`**:
```python
def import_module_with_full_path(file_path):
    base_filename = os.path.basename(file_path)
    module_name = os.path.splitext(base_filename)[0]
    module_spec = importlib.util.spec_from_file_location(module_name, file_path)
    imported_module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(imported_module)
    return imported_module
```


## 2. Data Preparation

Let’s assume you have:
- A **training dataset** named `train_data.csv`
- A **test dataset** named `test_data.csv`

Each row might look like:

| y0 (label) | Y1 | Y2 | ... | Y12 | X1 | X2 | ... | Xn |
|------------|----|----|-----|-----|----|----|-----|----|
| 0 or 1     | …  | …  | …   | …   | …  | …  | …   | …  |

Where:
- **y0** is your single-class label (e.g., 0 or 1)
- **Y** columns (Y1–Y12) are intermediate targets
- **X** columns (X1–Xn) are input features

### Load Your Data
```python
# 1) Import the BioMapAI module
BioMapAI = import_module_with_full_path("BioMapAI.py")

# 2) Load training data
data = pd.read_csv("example_data/train_data.csv", index_col=0)
y0 = data.iloc[:, 0]       # Single-class label
Y = data.iloc[:, 1:13]     # Intermediate target columns
X = data.iloc[:, 13:]      # Input features

# 3) Load test data
test_data = pd.read_csv("example_data/test_data.csv", index_col=0)
y0_test = test_data.iloc[:, 0]
Y_test = test_data.iloc[:, 1:13]
X_test = test_data.iloc[:, 13:]
```


## 3. Model Initialization and Training

### 3.1 **OmicScoreModel**
The **OmicScoreModel** is trained to predict the intermediate targets `Y`.

```python
model, model_history = BioMapAI.OmicScoreModel(
    epochs=500, 
    optimizer=tf.keras.optimizers.Adam(0.0005), 
    batch_size=64, 
    kernel_regularizer=tf.keras.regularizers.L2(0.008), 
    model_name="example"
).train(X, Y)
```
- **epochs**: Number of training iterations.
- **optimizer**: Choice of optimizer and learning rate.
- **batch_size**: Batch size for training.
- **kernel_regularizer**: Regularization to avoid overfitting.
- **model_name**: Useful if you want to save or track multiple models.

### 3.2 Monitoring the Training History
```python
print(model_history.history.keys())
# You can plot loss curves, accuracy, etc.
```


## 4. Prediction and Evaluation

### 4.1 Predict Intermediate Targets `Y`
```python
Y_pred = BioMapAI.OmicScoreModel().predict(model, X_test)
```
This outputs the intermediate predictions for each of the 12 `Y` columns.

### 4.2 Evaluate Intermediate Target Predictions
Compute the Mean Squared Error (MSE) on `Y_test`:
```python
mse = BioMapAI.OmicScoreModel().score(model, X_test, Y_test)
print("Intermediate Target MSE:", mse)
```


## 5. Score Conversion and Weight Adjustment

### 5.1 Build a Layer for Converting `Y` to `y0`
```python
Y_to_y_model = BioMapAI.ScoreLayer().build_model(Y, y0)
```
This creates a secondary model that maps the 12-dimensional `Y` to a single label or score.

### 5.2 Predict Final Labels `y0`
```python
y_pred = BioMapAI.ScoreYModel(model, Y_to_y_model).predict(X_test)
```
### 5.3 Evaluate Final Model
```python
loss, accuracy = BioMapAI.ScoreYModel(model, Y_to_y_model).evaluate(X_test, y0_test)
print("Final Model Loss:", loss)
print("Final Model Accuracy:", accuracy)
```

### 5.4 Adjusting Weights (Optional)
If you want to fine-tune how the intermediate predictions map to the final label:
```python
Y_to_y_model_adjust = BioMapAI.WeightsAdjust(model, X, Y, Y_to_y_model).adjust_score_weight()
```
Then re-predict and re-evaluate:
```python
y_pred = BioMapAI.ScoreYModel(model, Y_to_y_model).predict(X_test)
loss, accuracy = BioMapAI.ScoreYModel(model, Y_to_y_model).evaluate(X_test, y0_test)
print("Adjusted Final Model Loss:", loss)
print("Adjusted Final Model Accuracy:", accuracy)
```


## Putting It All Together

Below is a compact version of the entire flow. You can copy this code into its own file (e.g., `train_and_eval.py`) or run it as a Jupyter cell.


In [29]:
#!/usr/bin/env python
# coding: utf-8

import pandas as pd
import numpy as np
import random
import tensorflow as tf
tf.keras.backend.clear_session()

import importlib.util
import os

def import_module_with_full_path(file_path):
    base_filename = os.path.basename(file_path)
    module_name = os.path.splitext(base_filename)[0]
    module_spec = importlib.util.spec_from_file_location(module_name, file_path)
    imported_module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(imported_module)
    return imported_module

# 1) Import BioMapAI module
BioMapAI = import_module_with_full_path("BioMapAI.py")

# 2) Load data
data = pd.read_csv("example_data/train_data.csv", index_col=0)
y0 = data.iloc[:, 0]
Y = data.iloc[:, 1:13]
X = data.iloc[:, 13:]

test_data = pd.read_csv("example_data/test_data.csv", index_col=0)
y0_test = test_data.iloc[:, 0]
Y_test = test_data.iloc[:, 1:13]
X_test = test_data.iloc[:, 13:]

# 3) Train OmicScoreModel
model, model_history = BioMapAI.OmicScoreModel(
    epochs=5,
    optimizer=tf.keras.optimizers.Adam(0.0005),
    batch_size=64,
    kernel_regularizer=tf.keras.regularizers.L2(0.008),
    model_name="example"
).train(X, Y)

# 4) Predict and evaluate intermediate targets
Y_pred = BioMapAI.OmicScoreModel().predict(model, X_test)
mse = BioMapAI.OmicScoreModel().score(model, X_test, Y_test)
print("Intermediate Target MSE:", mse)

# 5) Build layer to convert Y --> y0
Y_to_y_model = BioMapAI.ScoreLayer().build_model(Y, y0)

# 6) Predict final output
y_pred = BioMapAI.ScoreYModel(model, Y_to_y_model).predict(X_test)

# 7) Evaluate final output
loss, accuracy = BioMapAI.ScoreYModel(model, Y_to_y_model).evaluate(X_test, y0_test)
print("Initial Final Model Loss:", loss)
print("Initial Final Model Accuracy:", accuracy)

# 8) (Optional) Adjust weights
Y_to_y_model_adjust = BioMapAI.WeightsAdjust(model, X, Y, Y_to_y_model).adjust_score_weight()

# 9) Predict & Evaluate again
y_pred = BioMapAI.ScoreYModel(model, Y_to_y_model).predict(X_test)
loss, accuracy = BioMapAI.ScoreYModel(model, Y_to_y_model).evaluate(X_test, y0_test)
print("Adjusted Final Model Loss:", loss)
print("Adjusted Final Model Accuracy:", accuracy)

FileNotFoundError: [Errno 2] No such file or directory: 'BioMapAI.py'

## Conclusion
You have now:
1. **Loaded and prepared** your data
2. **Trained** the OmicScoreModel on intermediate targets
3. **Transformed** those intermediate predictions to final labels
4. **Evaluated** and **(optionally) adjusted** model weights

Feel free to experiment with different architectures, hyperparameters (learning rate, batch size, etc.), or data splits. Modify **BioMapAI.py** to suit your specific needs.

**Happy modeling!**

In [ ]:
end_time = time.time()
print(f"Total execution time: {end_time - start_time:.2f} seconds")

Total execution time: 7.03 seconds


In [ ]:
import pandas as pd

meta1 = pd.read_csv(".../data/metadata/Metadata_061523.csv")
meta2 = pd.read_csv(".../data/metadata/Metadata_100322.csv")

print("Metadata_061523:", meta1.shape)
print(meta1.head())
print(meta1.columns.tolist())

print("\n====================\n")

print("Metadata_100322:", meta2.shape)
print(meta2.head())
print(meta2.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '.../data/metadata/Metadata_061523.csv'

In [ ]:
print(meta1["study_ptorhc"].value_counts())

print(meta1["timepoints"].value_counts())

print("Unique patients:")
print(meta1["samples_id"].nunique())

MECFS      348
Control    167
Name: study_ptorhc, dtype: int64
tp1    249
tp2    168
tp3     94
tp4      4
Name: timepoints, dtype: int64
Unique patients:
514


In [ ]:
print(meta1.columns.tolist())

['Unnamed: 0', 'sample_id_tp1', 'samples_id', 'timepoints', 'study_ptorhc', 'illness_duration', 'age', 'age_group', 'gender', 'bmi', 'ethnic', 'race', 'diet_meat', 'diet_sugar', 'diet_veg', 'diet_grains', 'diet_fruit', 'antifungals', 'antibiotics', 'probiotics', 'antivirals', 'IBS']


In [ ]:
metab = pd.read_csv("data/metabolomics/Metabolomics_maaslined_norun.csv")

print(metab.shape)
print(metab.head())
print(metab.columns[:10].tolist())

(876, 415)
                    Unnamed: 0  SAM000203_tp1  SAM000269_tp1  SAM000268_tp1  \
0  S-1-pyrroline-5-carboxylate       0.733745       1.476325       1.411830   
1         1-methylnicotinamide       0.382365       0.387559       1.383625   
2          alpha-ketoglutarate       0.847127       0.342713       0.350704   
3         3-hydroxyisobutyrate       0.442159       0.488864       0.187794   
4  3-hydroxy-3-methylglutarate       0.784203       0.907234       1.005262   

   SAM000266_tp1  SAM000257_tp1  SAM000260_tp1  SAM000259_tp1  SAM000253_tp1  \
0       0.566404       1.481165       0.496603       1.168407       0.585473   
1       0.840769       0.846519       0.478453       0.768354       0.928184   
2       0.464346       0.483156       0.431380       0.382185       0.400653   
3       1.405387       0.437858       1.225215       0.608739       0.072016   
4       1.436603       1.052833       0.852716       1.032662       1.818857   

   SAM000254_tp1  ...  SAM000536_

In [ ]:
print(meta1.columns.tolist())

['Unnamed: 0', 'sample_id_tp1', 'samples_id', 'timepoints', 'study_ptorhc', 'illness_duration', 'age', 'age_group', 'gender', 'bmi', 'ethnic', 'race', 'diet_meat', 'diet_sugar', 'diet_veg', 'diet_grains', 'diet_fruit', 'antifungals', 'antibiotics', 'probiotics', 'antivirals', 'IBS']


In [ ]:
print(meta1.head(1).T)

                              0
Unnamed: 0        SAM000201_tp1
sample_id_tp1         SAM000201
samples_id            SAM000201
timepoints                  tp1
study_ptorhc              MECFS
illness_duration          Short
age                          64
age_group                   >50
gender                   Female
bmi                   25.056806
ethnic             Not Hispanic
race                      White
diet_meat                   3.0
diet_sugar                  1.0
diet_veg                    4.0
diet_grains                 1.0
diet_fruit                  4.0
antifungals                  No
antibiotics                  No
probiotics                   No
antivirals                   No
IBS                         2.0


In [ ]:
print(meta1["Unnamed: 0"].tail())
metab = pd.read_csv(
    "data/metabolomics/Metabolomics_maaslined_norun.csv"
)

print(metab.columns[:500].tolist())

510    SAM000538_tp1
511    SAM000538_tp2
512    SAM000539_tp1
513    SAM000553_tp1
514    SAM000200_tp1
Name: Unnamed: 0, dtype: object


['Unnamed: 0', 'SAM000203_tp1', 'SAM000269_tp1', 'SAM000268_tp1', 'SAM000266_tp1', 'SAM000257_tp1', 'SAM000260_tp1', 'SAM000259_tp1', 'SAM000253_tp1', 'SAM000254_tp1', 'SAM000258_tp1', 'SAM000247_tp1', 'SAM000256_tp1', 'SAM000249_tp1', 'SAM000248_tp1', 'SAM000246_tp1', 'SAM000237_tp1', 'SAM000241_tp1', 'SAM000234_tp1', 'SAM000250_tp1', 'SAM000236_tp1', 'SAM000213_tp1', 'SAM000245_tp1', 'SAM000201_tp1', 'SAM000204_tp1', 'SAM000208_tp1', 'SAM000261_tp1', 'SAM000255_tp1', 'SAM000251_tp1', 'SAM000290_tp1', 'SAM000291_tp1', 'SAM000272_tp1', 'SAM000288_tp1', 'SAM000265_tp1', 'SAM000284_tp1', 'SAM000263_tp1', 'SAM000286_tp1', 'SAM000281_tp1', 'SAM000273_tp1', 'SAM000287_tp1', 'SAM000280_tp1', 'SAM000270_tp1', 'SAM000275_tp1', 'SAM000279_tp1', 'SAM000380_tp1', 'SAM000395_tp1', 'SAM000370_tp1', 'SAM000387_tp1', 'SAM000367_tp1', 'SAM000388_tp1', 'SAM000366_tp1', 'SAM000384_tp1', 'SAM000371_tp1', 'SAM000393_tp1', 'SAM000372_tp1', 'SAM000216_tp2', 'SAM000373_tp1', 'SAM000378_tp1', 'SAM000368_tp1',

In [ ]:
meta_ids = set(meta1["Unnamed: 0"])
metab_ids = set(metab.columns[1:])  # skip "Unnamed: 0"

print("Metadata samples:", len(meta_ids))
print("Metabolomics samples:", len(metab_ids))
print("Overlap:", len(meta_ids & metab_ids))

Metadata samples: 515
Metabolomics samples: 414
Overlap: 414


In [ ]:
species = pd.read_csv(
    "data/metagenomics/Specie_abundance_maaslined.csv"
)

species_ids = set(species.columns[1:])

print("Species samples:", len(species_ids))
print("Overlap with metadata:", len(meta_ids & species_ids))

Species samples: 479
Overlap with metadata: 479


In [ ]:
quest = pd.read_csv("data/quest_lab/Quest_residue.csv")

print(quest.shape)
print(quest.columns[:10].tolist())

(48, 504)
['Unnamed: 0', 'SAM000201_tp1', 'SAM000201_tp2', 'SAM000201_tp3', 'SAM000201_tp4', 'SAM000202_tp1', 'SAM000202_tp2', 'SAM000202_tp3', 'SAM000202_tp4', 'SAM000203_tp1']


In [ ]:
kegg = pd.read_csv("data/metagenomics/KEGG_maaslined.csv")

kegg_ids = set(kegg.columns[1:])

print("KEGG samples:", len(kegg_ids))
print("Overlap:", len(meta_ids & kegg_ids))
print("Shape:", kegg.shape)

KEGG samples: 479
Overlap: 479
Shape: (4429, 480)


In [ ]:
immune = pd.read_csv("data/immuno/Immune_residue.csv")

immune_ids = set(immune.columns[1:])

print("Immune samples:", len(immune_ids))
print("Overlap:", len(meta_ids & immune_ids))
print("Shape:", immune.shape)

Immune samples: 489
Overlap: 489
Shape: (311, 490)


In [ ]:
common = meta_ids & metab_ids & species_ids & kegg_ids & immune_ids

print("Patients with all major omics:", len(common))

Patients with all major omics: 384


In [ ]:
quest_ids = set(quest.columns[1:])

print("Quest overlap:", len(meta_ids & quest_ids))

all_common = (
    meta_ids
    & metab_ids
    & species_ids
    & kegg_ids
    & immune_ids
    & quest_ids
)

print("Patients with EVERYTHING:", len(all_common))

Quest overlap: 503
Patients with EVERYTHING: 380


In [ ]:
for col in meta1.columns:
    print(col)

Unnamed: 0
sample_id_tp1
samples_id
timepoints
study_ptorhc
illness_duration
age
age_group
gender
bmi
ethnic
race
diet_meat
diet_sugar
diet_veg
diet_grains
diet_fruit
antifungals
antibiotics
probiotics
antivirals
IBS


In [ ]:
immune = pd.read_csv("data/immuno/Immune_residue.csv")


In [ ]:
print(immune.head())
print(immune.shape)
immune.columns
immune.index

        Unnamed: 0  SAM000208_tp1  SAM000208_tp2  SAM000204_tp1  \
0   total % CD3 d0      11.618731       0.218731      12.432509   
1  total % CD4+ d0      12.420000       7.120000      -3.780000   
2  total % CD8+ d0     -10.890640      -6.790640       9.762666   
3       CD4:CD8 d0       3.177490       0.902730      -2.279986   
4    total % DN d0       0.974889       2.334889      -0.972634   

   SAM000209_tp1  SAM000209_tp2  SAM000209_tp3  SAM000202_tp1  SAM000202_tp2  \
0       4.412318       5.812318       6.912318       3.084762       2.184762   
1       0.320000       1.620000      -1.380000      -8.936364     -12.036364   
2       5.559612       3.428288       6.028288      12.990964      15.890964   
3      -1.724720      -1.412109      -1.822940      -1.468629      -1.629472   
4      -1.650839      -1.850839      -1.540839      -1.625793      -1.795793   

   SAM000202_tp3  ...  SAM000499_tp2  SAM000499_tp3  SAM000485_tp1  \
0       1.184762  ...       2.204953      -8.1

RangeIndex(start=0, stop=311, step=1)

In [ ]:
for col in meta1.columns:
    print(col)


Unnamed: 0
sample_id_tp1
samples_id
timepoints
study_ptorhc
illness_duration
age
age_group
gender
bmi
ethnic
race
diet_meat
diet_sugar
diet_veg
diet_grains
diet_fruit
antifungals
antibiotics
probiotics
antivirals
IBS


In [ ]:
print("Metadata:", immune.shape)
print("Metabolomics:", immune.shape)
print("Immune:", immune.shape)
print("KEGG:", immune.shape)
print("Species:", immune.shape)
print("Quest:", immune.shape)

Metadata: (311, 490)
Metabolomics: (311, 490)
Immune: (311, 490)
KEGG: (311, 490)
Species: (311, 490)
Quest: (311, 490)


In [ ]:
immuneP = pd.read_csv("data/immuno/Immune_percentage.csv")
print("Immune:", immuneP.shape)

Immune: (311, 490)


In [ ]:
immune = pd.read_csv("data/immuno/Immune_percentage.csv")

immune_ids = set(immune.columns[1:])

print("Immune samples:", len(immune_ids))
print("Overlap:", len(meta_ids & immune_ids))
print("Shape:", immune.shape)

Immune samples: 489
Overlap: 489
Shape: (311, 490)


In [ ]:
metab = pd.read_csv("data/metabolomics/Metabolomics_maaslined_norun.csv")

metab_ids = set(metab.columns[1:])

print("METAB samples:", len(metab_ids))
print("Overlap:", len(meta_ids & metab_ids))
print("Shape:", metab.shape)

METAB samples: 414
Overlap: 414
Shape: (876, 415)


In [ ]:
metab = pd.read_csv("data/metabolomics/Metabolomics_masslined.csv")

metab_ids = set(metab.columns[1:])

print("METAB samples:", len(metab_ids))
print("Overlap:", len(meta_ids & metab_ids))
print("Shape:", metab.shape)

METAB samples: 414
Overlap: 414
Shape: (395, 415)


In [ ]:
metab1 = pd.read_csv("data/metabolomics/Metabolomics_maaslined_norun.csv")
metab1.head()



,Unnamed: 0,SAM000203_tp1,SAM000269_tp1,SAM000268_tp1,SAM000266_tp1,SAM000257_tp1,SAM000260_tp1,SAM000259_tp1,SAM000253_tp1,SAM000254_tp1,...,SAM000536_tp2,SAM000529_tp2,SAM000283_tp3,SAM000500_tp2,SAM000250_tp3,SAM000306_tp3,SAM000309_tp3,SAM000470_tp2,SAM000458_tp2,SAM000304_tp3
0,S-1-pyrroline-5-carboxylate,0.733745,1.476325,1.411830,0.566404,1.481165,0.496603,1.168407,0.585473,0.964275,...,0.824462,0.928435,1.713793,0.723486,1.169712,0.839558,0.916199,0.482881,0.813109,0.171151
1,1-methylnicotinamide,0.382365,0.387559,1.383625,0.840769,0.846519,0.478453,0.768354,0.928184,0.700113,...,1.223317,1.011842,1.056749,0.498745,0.967518,0.982720,0.677429,1.120531,0.817388,2.692711
2,alpha-ketoglutarate,0.847127,0.342713,0.350704,0.464346,0.483156,0.431380,0.382185,0.400653,0.608075,...,0.308320,0.374151,0.330072,0.313287,0.345309,0.353886,0.486239,0.438559,0.555517,0.329638
3,3-hydroxyisobutyrate,0.442159,0.488864,0.187794,1.405387,0.437858,1.225215,0.608739,0.072016,0.072016,...,0.756751,0.072016,1.162334,0.971753,0.072016,0.072016,1.301529,0.072016,0.072016,0.072016
4,3-hydroxy-3-methylglutarate,0.784203,0.907234,1.005262,1.436603,1.052833,0.852716,1.032662,1.818857,1.370548,...,0.579216,0.811470,0.950055,0.817860,0.662079,0.839763,0.816733,1.090485,0.850739,0.765920


In [ ]:
metab2 = pd.read_csv("data/metabolomics/Metabolomics_masslined.csv")
metab2.head()

,Unnamed: 0,SAM000203_tp1,SAM000269_tp1,SAM000268_tp1,SAM000266_tp1,SAM000257_tp1,SAM000260_tp1,SAM000259_tp1,SAM000253_tp1,SAM000254_tp1,...,SAM000536_tp2,SAM000529_tp2,SAM000283_tp3,SAM000500_tp2,SAM000250_tp3,SAM000306_tp3,SAM000309_tp3,SAM000470_tp2,SAM000458_tp2,SAM000304_tp3
0,cholate,0.431694,0.341328,0.830513,1.130044,0.362826,0.378776,0.140340,1.457921,2.075634,...,0.151607,0.224771,7.879539,2.226149,0.137264,0.169468,0.966070,0.187118,0.262365,0.185924
1,linoleate (18:2n6),0.440198,0.918709,0.685851,1.293946,0.463149,1.210857,1.160790,1.176204,1.587719,...,0.311125,1.183181,0.320631,0.500565,0.131226,0.713235,1.800576,0.301638,1.701795,1.907225
2,quinolinate,0.801856,0.619263,0.846281,1.409837,0.477352,0.857344,0.660002,0.373901,1.255196,...,0.342085,0.579146,0.495432,0.704972,1.446672,0.401487,1.614332,0.567299,1.561215,0.340129
3,arginine,0.759626,1.081989,1.029119,1.029414,0.886748,0.556088,0.917243,0.679491,0.653137,...,0.651650,0.818285,1.133824,0.839122,0.783296,1.088171,1.380665,0.768067,0.772562,0.686814
4,3-(4-hydroxyphenyl)lactate,0.988096,0.836442,0.999942,1.364680,0.878536,0.543524,0.998191,1.241667,1.120322,...,0.885661,0.586286,0.753290,0.970019,0.407884,0.901711,0.388571,0.936896,1.009477,1.165592


In [ ]:
print(metab1.columns[:5])
print(metab2.columns[:5])

print(metab1.iloc[:5, :5])
print(metab2.iloc[:5, :5])
print(metab1.iloc[:10, 0])
print(metab2.iloc[:10, 0])

Index(['Unnamed: 0', 'SAM000203_tp1', 'SAM000269_tp1', 'SAM000268_tp1',
       'SAM000266_tp1'],
      dtype='object')
Index(['Unnamed: 0', 'SAM000203_tp1', 'SAM000269_tp1', 'SAM000268_tp1',
       'SAM000266_tp1'],
      dtype='object')
                    Unnamed: 0  SAM000203_tp1  SAM000269_tp1  SAM000268_tp1  \
0  S-1-pyrroline-5-carboxylate       0.733745       1.476325       1.411830   
1         1-methylnicotinamide       0.382365       0.387559       1.383625   
2          alpha-ketoglutarate       0.847127       0.342713       0.350704   
3         3-hydroxyisobutyrate       0.442159       0.488864       0.187794   
4  3-hydroxy-3-methylglutarate       0.784203       0.907234       1.005262   

   SAM000266_tp1  
0       0.566404  
1       0.840769  
2       0.464346  
3       1.405387  
4       1.436603  
                   Unnamed: 0  SAM000203_tp1  SAM000269_tp1  SAM000268_tp1  \
0                     cholate       0.431694       0.341328       0.830513   
1          linole

In [ ]:
annot = pd.read_csv(
    "data/metabolomics/QC-norm Data Common wImp_Jan_06_2022.csv"
)

print(annot.shape)

(957, 432)


In [ ]:
print(annot.columns[:20])

Index(['CHEM_ID', 'SUPER_PATHWAY', 'SUB_PATHWAY', 'PATHWAY_SORTORDER', 'TYPE',
       'INCHIKEY', 'SMILES', 'CHEMICAL_NAME', 'PLOT_NAME', 'CAS', 'CHEMSPIDER',
       'HMDB', 'KEGG', 'PUBCHEM', 'CHRO_LIB_ENTRY_ID', 'COMP_ID', 'LIB_ID',
       'PLATFORM', 'SAM000203_tp1', 'SAM000269_tp1'],
      dtype='object')


In [ ]:
print(annot.columns[-20:])

Index(['SAM000320_tp3', 'SAM000286_tp3', 'SAM000201_tp4', 'SAM000524_tp2',
       'SAM000538_tp2', 'SAM000308_tp3', 'SAM000312_tp3', 'SAM000475_tp2',
       'SAM000225_tp3', 'SAM000537_tp2', 'SAM000536_tp2', 'SAM000529_tp2',
       'SAM000283_tp3', 'SAM000500_tp2', 'SAM000250_tp3', 'SAM000306_tp3',
       'SAM000309_tp3', 'SAM000470_tp2', 'SAM000458_tp2', 'SAM000304_tp3'],
      dtype='object')


In [ ]:
annotation_cols = 18
print(len(annot.columns[annotation_cols:]))

414


In [ ]:
sample_ids = set(annot.columns[18:])
print(len(meta_ids & sample_ids))

414


In [ ]:
import pandas as pd
import os

# -----------------------------
# Load metadata
# -----------------------------
meta = pd.read_csv("data/metadata/Metadata_061523.csv")
meta_ids = set(meta.iloc[:,0])

print(f"Metadata: {meta.shape}")
print()

# -----------------------------
# Files to inspect
# -----------------------------
files = {
    "Metabolomics NO RUN":
        "data/metabolomics/Metabolomics_maaslined_norun.csv",

    "Metabolomics":
        "data/metabolomics/Metabolomics_masslined.csv",

    "QC-Norm Data":
        "data/metabolomics/QC-norm Data Common wImp_Jan_06_2022.csv",

    "Species":
        "data/metagenomics/Specie_abundance_maaslined.csv",

    "KEGG":
        "data/metagenomics/KEGG_maaslined.csv",

    "Immune (Residue)":
        "data/immuno/Immune_residue.csv",

    "Immune (%)":
        "data/immuno/Immune_percentage.csv",

    "Quest":
        "data/quest_lab/Quest_labs.csv"
}

# -----------------------------
# Analyze each file
# -----------------------------
for name, path in files.items():

    print("="*60)
    print(name)

    try:
        df = pd.read_csv(path)

        print("Shape:", df.shape)

        # Count metadata columns before sample IDs start
        sample_start = 0
        for i, col in enumerate(df.columns):
            if str(col).startswith("SAM"):
                sample_start = i
                break

        sample_ids = set(df.columns[sample_start:])

        print("Sample columns:", len(sample_ids))
        print("Metadata overlap:", len(meta_ids & sample_ids))
        print("Annotation columns:", sample_start)

        print("\nFirst 10 columns:")
        print(list(df.columns[:10]))

        print()

    except Exception as e:
        print("ERROR:", e)
        print()

summary = []

for name, path in files.items():

    try:
        df = pd.read_csv(path)

        sample_start = 0
        for i, col in enumerate(df.columns):
            if str(col).startswith("SAM"):
                sample_start = i
                break

        sample_ids = set(df.columns[sample_start:])

        summary.append({
            "Modality": name,
            "Rows": df.shape[0],
            "Columns": df.shape[1],
            "Sample Columns": len(sample_ids),
            "Annotation Columns": sample_start,
            "Metadata Overlap": len(meta_ids & sample_ids)
        })

    except:
        pass

summary_df = pd.DataFrame(summary)
summary_df

FileNotFoundError: [Errno 2] No such file or directory: 'data/metadata/Metadata_061523.csv'

In [ ]:
import pandas as pd
from pathlib import Path

# metadata IDs
meta = pd.read_csv("data/metadata/Metadata_061523.csv")
meta_ids = set(meta.iloc[:, 0].astype(str))

rows = []

for path in Path("data").rglob("*"):
    if path.suffix.lower() not in [".csv", ".txt"]:
        continue
    
    try:
        df = pd.read_csv(path)
    except:
        try:
            df = pd.read_csv(path, sep="\t")
        except Exception as e:
            rows.append({
                "File": str(path),
                "Rows": "ERROR",
                "Columns": "ERROR",
                "Sample Columns": "ERROR",
                "Annotation Columns": "ERROR",
                "Metadata Overlap": "ERROR",
                "Error": str(e)
            })
            continue

    # find first sample column
    sample_start = None
    for i, col in enumerate(df.columns):
        if str(col).startswith("SAM"):
            sample_start = i
            break

    if sample_start is not None:
        sample_cols = list(df.columns[sample_start:])
        sample_ids = set(map(str, sample_cols))
        annotation_cols = sample_start
    else:
        sample_cols = []
        sample_ids = set()
        annotation_cols = df.shape[1]

    rows.append({
        "File": str(path),
        "Folder": str(path.parent),
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Sample Columns": len(sample_cols),
        "Annotation Columns": annotation_cols,
        "Metadata Overlap": len(meta_ids & sample_ids),
        "First 5 Columns": list(df.columns[:5])
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["Folder", "File"]).reset_index(drop=True)
summary_df

FileNotFoundError: [Errno 2] No such file or directory: 'data/metadata/Metadata_061523.csv'

In [ ]:
datasets = {
    "Metadata": meta_ids,
    "Metabolomics": metab_ids,
    "Species": species_ids,
    "KEGG": kegg_ids,
    "Immune": immune_ids,
    "Quest": quest_ids
}

common = set.intersection(*datasets.values())

print("Samples in every dataset:", len(common))

NameError: name 'meta_ids' is not defined

In [ ]:
for name, ids in datasets.items():
    print(
        f"{name}: "
        f"{len(common & ids)}/{len(ids)} "
        f"({len(ids - common)} missing)"
    )

Metadata: 380/515 (135 missing)
Metabolomics: 380/414 (34 missing)
Species: 380/479 (99 missing)
KEGG: 380/479 (99 missing)
Immune: 380/489 (109 missing)
Quest: 380/503 (123 missing)


In [ ]:
meta1 = pd.read_csv("data/metadata/Metadata_100322.csv")
meta_ids1 = set(meta.iloc[:, 0].astype(str))
print(meta1.head())


      Unnamed: 0 sample_id_tp1   samples_id timepoints study_ptorhc  \
0  SAM000201_tp1     SAM000201    SAM000201        tp1        MECFS   
1  SAM000201_tp2     SAM000201  SAM000201_2        tp2        MECFS   
2  SAM000201_tp3     SAM000201    SAM000531        tp3        MECFS   
3  SAM000201_tp4     SAM000201    SAM000677        tp4        MECFS   
4  SAM000202_tp1     SAM000202  SAM000202_1        tp1        MECFS   

  illness_duration  age age_group  gender        ethnic  ... diet_meat  \
0            Short   64       >50  Female  Not Hispanic  ...       3.0   
1            Short   64       >50  Female  Not Hispanic  ...       3.0   
2            Short   64       >50  Female  Not Hispanic  ...       3.0   
3            Short   64       >50  Female  Not Hispanic  ...       3.0   
4            Short   45       <50    Male  Not Hispanic  ...       3.0   

   diet_sugar  diet_veg  diet_grains  diet_fruit  antifungals antibiotics  \
0         1.0       4.0          1.0         4.0   

In [ ]:
meta2 = pd.read_csv("data/metadata/Metadata_061523.csv")
meta_ids2 = set(meta.iloc[:, 0].astype(str))
print(meta2.head())


      Unnamed: 0 sample_id_tp1   samples_id timepoints study_ptorhc  \
0  SAM000201_tp1     SAM000201    SAM000201        tp1        MECFS   
1  SAM000201_tp2     SAM000201  SAM000201_2        tp2        MECFS   
2  SAM000201_tp3     SAM000201    SAM000531        tp3        MECFS   
3  SAM000201_tp4     SAM000201    SAM000677        tp4        MECFS   
4  SAM000202_tp1     SAM000202  SAM000202_1        tp1        MECFS   

  illness_duration  age age_group  gender        bmi  ... diet_meat  \
0            Short   64       >50  Female  25.056806  ...       3.0   
1            Short   64       >50  Female  25.056806  ...       3.0   
2            Short   64       >50  Female  25.056806  ...       3.0   
3            Short   64       >50  Female  25.056806  ...       3.0   
4            Short   45       <50    Male  25.358989  ...       3.0   

  diet_sugar  diet_veg  diet_grains  diet_fruit  antifungals  antibiotics  \
0        1.0       4.0          1.0         4.0           No         

In [ ]:
print(meta1.iloc[0])


Unnamed: 0          SAM000201_tp1
sample_id_tp1           SAM000201
samples_id              SAM000201
timepoints                    tp1
study_ptorhc                MECFS
illness_duration            Short
age                            64
age_group                     >50
gender                     Female
ethnic               Not Hispanic
race                        White
diet_meat                     3.0
diet_sugar                    1.0
diet_veg                      4.0
diet_grains                   1.0
diet_fruit                    4.0
antifungals                    No
antibiotics                    No
probiotics                     No
antivirals                     No
IBS                           2.0
Name: 0, dtype: object


print(meta2.iloc[0])

In [ ]:
print(meta1.shape)

(515, 21)


In [ ]:
print(meta2.shape)

(515, 22)


In [ ]:
print(meta1["Unnamed: 0"])

0      SAM000201_tp1
1      SAM000201_tp2
2      SAM000201_tp3
3      SAM000201_tp4
4      SAM000202_tp1
           ...      
510    SAM000538_tp1
511    SAM000538_tp2
512    SAM000539_tp1
513    SAM000553_tp1
514    SAM000200_tp1
Name: Unnamed: 0, Length: 515, dtype: object


In [ ]:
set(meta_ids1)== set(meta_ids2)

True

In [ ]:
print(meta1["Unnamed: 0"].equals(meta2["Unnamed: 0"]))

True


In [ ]:
meta2 = pd.read_csv("data/metadata/Metadata_061523.csv")
# meta_ids2 = set(meta.iloc[:, 0].astype(str))
print(meta2.iloc[:, 1])
print(meta2.iloc[:, 1].nunique())

0      SAM000201
1      SAM000201
2      SAM000201
3      SAM000201
4      SAM000202
         ...    
510    SAM000538
511    SAM000538
512    SAM000539
513    SAM000553
514    SAM000200
Name: sample_id_tp1, Length: 515, dtype: object
249


In [ ]:
meta2 = pd.read_csv("data/metadata/Metadata_061523.csv")
# meta_ids2 = set(meta.iloc[:, 0].astype(str))
print(meta2.iloc[:, 3])
print(meta2.iloc[:, 3].value_counts())

0      tp1
1      tp2
2      tp3
3      tp4
4      tp1
      ... 
510    tp1
511    tp2
512    tp1
513    tp1
514    tp1
Name: timepoints, Length: 515, dtype: object
tp1    249
tp2    168
tp3     94
tp4      4
Name: timepoints, dtype: int64


In [ ]:
meta2 = pd.read_csv("data/metadata/Metadata_061523.csv")
# meta_ids2 = set(meta.iloc[:, 0].astype(str))
print(meta2.iloc[:, 1])
print(meta2.iloc[:, 1].value_counts().value_counts())

0      SAM000201
1      SAM000201
2      SAM000201
3      SAM000201
4      SAM000202
         ...    
510    SAM000538
511    SAM000538
512    SAM000539
513    SAM000553
514    SAM000200
Name: sample_id_tp1, Length: 515, dtype: object
3    88
1    79
2    78
4     4
Name: sample_id_tp1, dtype: int64


In [ ]:
patterns = (
    meta2
    .groupby("sample_id_tp1")["timepoints"]
    .apply(lambda x: "+".join(sorted(x)))
)

print(patterns.head())

sample_id_tp1
SAM000200                tp1
SAM000201    tp1+tp2+tp3+tp4
SAM000202    tp1+tp2+tp3+tp4
SAM000203            tp1+tp2
SAM000204                tp1
Name: timepoints, dtype: object


In [30]:
cohort_table = pd.DataFrame(patterns)

cohort_table["visits"] = meta2.groupby("sample_id_tp1")["timepoints"].count()
patient_info = meta2.groupby("sample_id_tp1").first()
cohort_table["disease"] = patient_info["study_ptorhc"]
cohort_table["illness_duration"] = patient_info["illness_duration"]
cohort_table["age"] = patient_info["age"]
cohort_table["age_group"] = patient_info["age_group"]
cohort_table["gender"] = patient_info["gender"]
cohort_table["bmi"] = patient_info["bmi"]
cohort_table["ethnic"] = patient_info["ethnic"]
cohort_table["race"] = patient_info["race"]
print(cohort_table)

NameError: name 'patterns' is not defined

In [ ]:
pd.crosstab(cohort_table["timepoints"], cohort_table["disease"])

disease,Control,MECFS
timepoints,,
tp1,38,41
tp1+tp2,45,31
tp1+tp2+tp3,13,75
tp1+tp2+tp3+tp4,0,4
tp1+tp3,0,2


In [ ]:
cohort_table.columns

Index(['timepoints', 'visits', 'disease', 'illness_duration', 'age',
       'age_group', 'gender', 'bmi', 'ethnic', 'race'],
      dtype='object')

In [ ]:
pd.crosstab(cohort_table["visits"], cohort_table["disease"])

disease,Control,MECFS
visits,,
1,38,41
2,45,33
3,13,75
4,0,4


In [ ]:
cohort_table["disease"].value_counts()

MECFS      153
Control     96
Name: disease, dtype: int64

In [ ]:
cohort_table.groupby("disease")[["age", "bmi"]].describe()

age                                                         bmi  \
         count       mean        std   min   25%   50%    75%   max  count   
disease                                                                      
Control   96.0  40.593750  13.480261  20.0  29.0  37.0  51.25  67.0   75.0   
MECFS    153.0  44.444444  13.288741  19.0  33.0  44.0  56.00  68.0  135.0   

                                                                          \
              mean       std        min        25%        50%        75%   
disease                                                                    
Control  26.580617  5.649046  16.242386  22.292191  24.389796  30.130237   
MECFS    26.480829  5.766755  17.304615  21.820291  25.537551  29.865599   

                    
               max  
disease             
Control  42.767447  
MECFS    41.008333

In [ ]:
pd.crosstab(cohort_table["gender"], cohort_table["disease"])


disease,Control,MECFS
gender,,
Female,58,112
Male,38,41


In [ ]:
pd.crosstab(cohort_table["illness_duration"], cohort_table["disease"])

disease,Control,MECFS
illness_duration,,
Control,96,0
Long,0,78
Short,0,75


In [ ]:
import os

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
cohort_table.to_csv("../data/processed/cohort_table.csv")

NameError: name 'cohort_table' is not defined